# 06 - E5 Validacion holdout y sanity checks

**Objetivo:** validar de forma mas exigente el baseline binario mejorado, verificando que el buen desempeno no dependa solo del split interno usado en `05_E5_baseline_mejorado_binario.ipynb`.

Este notebook carga el modelo mejorado ya entrenado y lo evalua sobre casos T1 no utilizados en el entrenamiento ni en la validacion interna.

**Fuera de alcance:** reentrenamiento, cambio de arquitectura, multiclase, axial, nnU-Net e integracion backend.

## 1. Instalacion e importacion de dependencias

In [ ]:
!pip -q install SimpleITK scikit-image tqdm

In [ ]:
from pathlib import Path
import json
import random
import warnings

import SimpleITK as sitk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from skimage.transform import resize
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("SimpleITK:", sitk.Version())
print("CUDA disponible:", torch.cuda.is_available())
print("Dispositivo:", DEVICE)

## 2. Montaje de Google Drive y definicion de rutas

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
DATASET_ROOT = Path("/content/drive/MyDrive/PFI_MVP/data/SPIDER")
PREPROCESS_ROOT = Path("/content/drive/MyDrive/PFI_MVP/results/E4_preprocesamiento")
IMPROVED_ROOT = Path("/content/drive/MyDrive/PFI_MVP/results/E5_baseline_mejorado_binario")
HOLDOUT_ROOT = Path("/content/drive/MyDrive/PFI_MVP/results/E5_holdout_validacion")
FIGURES_ROOT = Path("/content/drive/MyDrive/PFI_MVP/figures")
DOCS_ROOT = Path("/content/drive/MyDrive/PFI_MVP/docs")

for path in [HOLDOUT_ROOT, FIGURES_ROOT, DOCS_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

CANDIDATES_CSV = PREPROCESS_ROOT / "E4_baseline_candidates_no_space.csv"
IMPROVED_SELECTED_CASES_CSV = IMPROVED_ROOT / "E5_improved_selected_cases.csv"
IMPROVED_MODEL_PATH = IMPROVED_ROOT / "E5_improved_unet2d_binary_best.pt"
IMPROVED_REPORT_JSON = IMPROVED_ROOT / "E5_improved_validation_report.json"

print("DATASET_ROOT:", DATASET_ROOT)
print("PREPROCESS_ROOT:", PREPROCESS_ROOT)
print("IMPROVED_ROOT:", IMPROVED_ROOT)
print("HOLDOUT_ROOT:", HOLDOUT_ROOT)
print("FIGURES_ROOT:", FIGURES_ROOT)
print("DOCS_ROOT:", DOCS_ROOT)

## 3. Cargar modelo mejorado

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class SimpleUNet2D(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, base_channels=16):
        super().__init__()
        self.enc1 = DoubleConv(in_channels, base_channels)
        self.enc2 = DoubleConv(base_channels, base_channels * 2)
        self.enc3 = DoubleConv(base_channels * 2, base_channels * 4)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(base_channels * 4, base_channels * 8)
        self.up3 = nn.ConvTranspose2d(base_channels * 8, base_channels * 4, 2, stride=2)
        self.dec3 = DoubleConv(base_channels * 8, base_channels * 4)
        self.up2 = nn.ConvTranspose2d(base_channels * 4, base_channels * 2, 2, stride=2)
        self.dec2 = DoubleConv(base_channels * 4, base_channels * 2)
        self.up1 = nn.ConvTranspose2d(base_channels * 2, base_channels, 2, stride=2)
        self.dec1 = DoubleConv(base_channels * 2, base_channels)
        self.out = nn.Conv2d(base_channels, out_channels, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b = self.bottleneck(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out(d1)


if not IMPROVED_MODEL_PATH.exists():
    raise FileNotFoundError(f"No existe el modelo mejorado: {IMPROVED_MODEL_PATH}")

checkpoint = torch.load(IMPROVED_MODEL_PATH, map_location=DEVICE)
BASE_CHANNELS = int(checkpoint.get("base_channels", 16))
TARGET_SIZE = tuple(checkpoint.get("target_size", (256, 256)))
SAGITTAL_AXIS = int(checkpoint.get("sagittal_axis", 2))

model = SimpleUNet2D(base_channels=BASE_CHANNELS).to(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Modelo cargado:", IMPROVED_MODEL_PATH)
print("base_channels:", BASE_CHANNELS)
print("target_size:", TARGET_SIZE)
print("sagittal_axis:", SAGITTAL_AXIS)
print("checkpoint keys:", checkpoint.keys())

## 4. Construir holdout T1 no usado

In [ ]:
N_HOLDOUT = 40
MODALITY_FILTER = "t1"
THRESHOLDS = [0.3, 0.4, 0.5, 0.6, 0.7]

candidates_df = pd.read_csv(CANDIDATES_CSV)
used_cases_df = pd.read_csv(IMPROVED_SELECTED_CASES_CSV)

def infer_case_modality(row):
    text = " ".join(str(v).lower() for v in row.values)
    if "t2" in text:
        return "t2"
    if "t1" in text:
        return "t1"
    return "unknown"


def ensure_case_id(df):
    out = df.copy()
    if "case_id" not in out.columns:
        for candidate in ["file_name", "image_path", "source_image_path"]:
            if candidate in out.columns:
                out["case_id"] = out[candidate].apply(lambda x: Path(str(x)).stem)
                break
    if "case_id" not in out.columns:
        raise ValueError("No se pudo construir case_id.")
    out["case_id"] = out["case_id"].astype(str)
    return out


candidates_df = ensure_case_id(candidates_df)
used_cases_df = ensure_case_id(used_cases_df)

if "modality" not in candidates_df.columns:
    candidates_df["modality"] = candidates_df.apply(infer_case_modality, axis=1)

t1_candidates_df = candidates_df[candidates_df["modality"].str.lower().eq(MODALITY_FILTER)].copy()
used_case_ids = set(used_cases_df["case_id"].astype(str))
holdout_pool_df = t1_candidates_df[~t1_candidates_df["case_id"].astype(str).isin(used_case_ids)].copy()

if len(holdout_pool_df) == 0:
    raise RuntimeError("No quedan casos T1 disponibles para holdout despues de excluir los usados en notebook 05.")

holdout_df = holdout_pool_df.sample(n=min(N_HOLDOUT, len(holdout_pool_df)), random_state=SEED).reset_index(drop=True)
holdout_selected_cases_csv_path = HOLDOUT_ROOT / "E5_holdout_selected_cases.csv"
holdout_df.to_csv(holdout_selected_cases_csv_path, index=False)

print("Candidatos totales:", len(candidates_df))
print("Candidatos T1:", len(t1_candidates_df))
print("Casos usados notebook 05:", len(used_case_ids))
print("Disponibles para holdout:", len(holdout_pool_df))
print("Evaluados en holdout:", len(holdout_df))
print("CSV holdout:", holdout_selected_cases_csv_path)
display(holdout_df.head())

## 5. Preparacion de datos

Se usa el mismo preprocesamiento del notebook 05: lectura `.mha`, normalizacion p1-p99, eje sagital 2, mascara binaria `mask > 0` y resize a `256x256`.

**Limitacion importante:** la seleccion del slice representativo sigue usando la mascara para elegir el corte con mayor area. Esto es aceptable para validar el baseline 2D, pero debe corregirse antes de un flujo de inferencia real.

In [ ]:
def resolve_path(value):
    return Path(str(value))


def get_case_paths(row):
    image_candidates = ["image_path", "source_image_path", "image", "img_path"]
    mask_candidates = ["mask_path", "source_mask_path", "mask", "seg_path"]
    image_path = None
    mask_path = None
    for column in image_candidates:
        if column in row and pd.notna(row[column]):
            image_path = resolve_path(row[column])
            break
    for column in mask_candidates:
        if column in row and pd.notna(row[column]):
            mask_path = resolve_path(row[column])
            break
    if image_path is None or mask_path is None:
        raise ValueError("No se encontraron columnas de path para imagen/mascara.")
    return image_path, mask_path


def read_mha(path: Path):
    itk_image = sitk.ReadImage(str(path))
    array = sitk.GetArrayFromImage(itk_image)
    return itk_image, array


def robust_percentile_normalize(image_array, p_low=1, p_high=99, eps=1e-8):
    image_float = image_array.astype(np.float32)
    low, high = np.percentile(image_float, [p_low, p_high])
    clipped = np.clip(image_float, low, high)
    if np.isclose(high, low):
        return np.zeros_like(clipped, dtype=np.float32)
    return ((clipped - low) / (high - low + eps)).astype(np.float32)


def representative_slice_index(mask_array, axis=2):
    if np.count_nonzero(mask_array) == 0:
        return int(mask_array.shape[axis] // 2)
    reduce_axes = tuple(ax for ax in range(mask_array.ndim) if ax != axis)
    area_by_slice = np.sum(mask_array > 0, axis=reduce_axes)
    return int(np.argmax(area_by_slice))


def take_slice(array, axis, index):
    return np.take(array, indices=index, axis=axis)


def resize_slice(array_2d, target_size=TARGET_SIZE, order=1):
    return resize(
        array_2d,
        output_shape=target_size,
        order=order,
        preserve_range=True,
        anti_aliasing=(order != 0),
    ).astype(np.float32)


def preprocess_case(row, axis=SAGITTAL_AXIS, target_size=TARGET_SIZE):
    image_path, mask_path = get_case_paths(row)
    itk_image, image = read_mha(image_path)
    itk_mask, mask = read_mha(mask_path)

    if image.shape != mask.shape:
        raise ValueError(f"Shape incompatible: image={image.shape}, mask={mask.shape}")

    image_norm = robust_percentile_normalize(image)
    mask_bin = (mask > 0).astype(np.float32)
    slice_index = representative_slice_index(mask_bin, axis=axis)

    image_slice = take_slice(image_norm, axis, slice_index)
    mask_slice = take_slice(mask_bin, axis, slice_index)

    image_slice = resize_slice(image_slice, target_size=target_size, order=1)
    mask_slice = resize_slice(mask_slice, target_size=target_size, order=0)
    mask_slice = (mask_slice > 0.5).astype(np.float32)

    return {
        "case_id": str(row["case_id"]),
        "image": image_slice[None, ...].astype(np.float32),
        "mask": mask_slice[None, ...].astype(np.float32),
        "slice_index": int(slice_index),
        "source_image_path": str(image_path),
        "source_mask_path": str(mask_path),
    }


class HoldoutDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe.reset_index(drop=True)
        self.cache = {}

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        if index not in self.cache:
            self.cache[index] = preprocess_case(self.dataframe.iloc[index])
        item = self.cache[index]
        return {
            "image": torch.from_numpy(item["image"]).float(),
            "mask": torch.from_numpy(item["mask"]).float(),
            "case_id": item["case_id"],
            "slice_index": item["slice_index"],
            "source_image_path": item["source_image_path"],
            "source_mask_path": item["source_mask_path"],
        }


holdout_dataset = HoldoutDataset(holdout_df)
holdout_loader = DataLoader(holdout_dataset, batch_size=1, shuffle=False, num_workers=0)

preview = next(iter(holdout_loader))
print("Image shape:", preview["image"].shape)
print("Mask shape:", preview["mask"].shape)
print("Mask unique:", torch.unique(preview["mask"]))

## 6. Inferencia y metricas por caso

In [ ]:
def dice_iou_numpy(pred_mask, gt_mask, eps=1e-7):
    pred = pred_mask.astype(bool)
    gt = gt_mask.astype(bool)
    intersection = np.logical_and(pred, gt).sum()
    pred_sum = pred.sum()
    gt_sum = gt.sum()
    union = np.logical_or(pred, gt).sum()
    dice = (2.0 * intersection + eps) / (pred_sum + gt_sum + eps)
    iou = (intersection + eps) / (union + eps)
    return float(dice), float(iou)


inference_items = []
metric_rows = []

model.eval()
with torch.no_grad():
    for batch in tqdm(holdout_loader, desc="Inferencia holdout"):
        images = batch["image"].to(DEVICE)
        masks = batch["mask"].to(DEVICE)
        logits = model(images)
        probs = torch.sigmoid(logits)

        image_np = images.detach().cpu().numpy()[0, 0]
        gt_np = masks.detach().cpu().numpy()[0, 0] > 0.5
        prob_np = probs.detach().cpu().numpy()[0, 0]

        inference_items.append({
            "case_id": batch["case_id"][0],
            "slice_index": int(batch["slice_index"][0]),
            "image": image_np,
            "gt": gt_np.astype(np.float32),
            "prob": prob_np,
        })

        for threshold in THRESHOLDS:
            pred_np = prob_np >= threshold
            dice, iou = dice_iou_numpy(pred_np, gt_np)
            metric_rows.append({
                "case_id": batch["case_id"][0],
                "slice_index": int(batch["slice_index"][0]),
                "threshold": float(threshold),
                "dice": dice,
                "iou": iou,
                "gt_foreground_ratio": float(gt_np.mean()),
                "pred_foreground_ratio": float(pred_np.mean()),
                "gt_positive_pixels": int(gt_np.sum()),
                "pred_positive_pixels": int(pred_np.sum()),
            })

holdout_metrics_df = pd.DataFrame(metric_rows)
holdout_metrics_csv_path = HOLDOUT_ROOT / "E5_holdout_metrics_by_case.csv"
holdout_metrics_df.to_csv(holdout_metrics_csv_path, index=False)

print("CSV metricas por caso:", holdout_metrics_csv_path)
display(holdout_metrics_df.head())

## 7. Resumen por threshold

In [ ]:
holdout_summary_df = (
    holdout_metrics_df.groupby("threshold", as_index=False)
    .agg(
        dice_mean=("dice", "mean"),
        dice_std=("dice", "std"),
        iou_mean=("iou", "mean"),
        iou_std=("iou", "std"),
        gt_foreground_ratio_mean=("gt_foreground_ratio", "mean"),
        pred_foreground_ratio_mean=("pred_foreground_ratio", "mean"),
        gt_positive_pixels_mean=("gt_positive_pixels", "mean"),
        pred_positive_pixels_mean=("pred_positive_pixels", "mean"),
    )
)

best_threshold = float(holdout_summary_df.loc[holdout_summary_df["dice_mean"].idxmax(), "threshold"])
th05_row = holdout_summary_df[np.isclose(holdout_summary_df["threshold"], 0.5)].iloc[0]
best_row = holdout_summary_df[np.isclose(holdout_summary_df["threshold"], best_threshold)].iloc[0]

holdout_summary_csv_path = HOLDOUT_ROOT / "E5_holdout_threshold_summary.csv"
holdout_summary_df.to_csv(holdout_summary_csv_path, index=False)

print("CSV resumen holdout:", holdout_summary_csv_path)
print("Mejor threshold por Dice:", best_threshold)
print("Dice th 0.5:", float(th05_row["dice_mean"]))
print("IoU th 0.5:", float(th05_row["iou_mean"]))
print("Dice mejor threshold:", float(best_row["dice_mean"]))
print("IoU mejor threshold:", float(best_row["iou_mean"]))
display(holdout_summary_df)

## 8. Sanity checks

In [ ]:
th05_metrics = holdout_metrics_df[np.isclose(holdout_metrics_df["threshold"], 0.5)].copy()

masks_are_binary = True
for item in inference_items:
    unique_values = set(np.unique(item["gt"]).tolist())
    masks_are_binary = masks_are_binary and unique_values.issubset({0.0, 1.0})

empty_prediction_cases = th05_metrics[th05_metrics["pred_positive_pixels"] == 0]["case_id"].tolist()
full_prediction_cases = th05_metrics[th05_metrics["pred_foreground_ratio"] > 0.999]["case_id"].tolist()

pct_pred_fg_gt_08 = float((th05_metrics["pred_foreground_ratio"] > 0.8).mean())
pct_pred_fg_lt_001 = float((th05_metrics["pred_foreground_ratio"] < 0.01).mean())

sanity_checks = {
    "masks_are_binary": bool(masks_are_binary),
    "empty_prediction_cases_threshold_05": empty_prediction_cases,
    "full_prediction_cases_threshold_05": full_prediction_cases,
    "pct_cases_pred_foreground_ratio_gt_0_8_threshold_05": pct_pred_fg_gt_08,
    "pct_cases_pred_foreground_ratio_lt_0_01_threshold_05": pct_pred_fg_lt_001,
    "has_empty_predictions_threshold_05": len(empty_prediction_cases) > 0,
    "has_full_predictions_threshold_05": len(full_prediction_cases) > 0,
}

sanity_checks_json_path = HOLDOUT_ROOT / "E5_holdout_sanity_checks.json"
sanity_checks_json_path.write_text(json.dumps(sanity_checks, indent=2), encoding="utf-8")

print("Sanity checks:", sanity_checks_json_path)
print(json.dumps(sanity_checks, indent=2))

## 9. Comparacion contra validacion interna

In [ ]:
if not IMPROVED_REPORT_JSON.exists():
    raise FileNotFoundError(f"No existe reporte mejorado: {IMPROVED_REPORT_JSON}")

improved_report = json.loads(IMPROVED_REPORT_JSON.read_text(encoding="utf-8"))

comparison = {
    "internal_dice_threshold_05": float(improved_report.get("dice_threshold_05", np.nan)),
    "internal_iou_threshold_05": float(improved_report.get("iou_threshold_05", np.nan)),
    "holdout_dice_threshold_05": float(th05_row["dice_mean"]),
    "holdout_iou_threshold_05": float(th05_row["iou_mean"]),
    "internal_pred_foreground_ratio_threshold_05": float(improved_report.get("pred_foreground_ratio_threshold_05", np.nan)),
    "holdout_pred_foreground_ratio_threshold_05": float(th05_row["pred_foreground_ratio_mean"]),
    "holdout_gt_foreground_ratio_threshold_05": float(th05_row["gt_foreground_ratio_mean"]),
    "internal_best_threshold": float(improved_report.get("best_threshold", np.nan)),
    "holdout_best_threshold": float(best_threshold),
    "internal_dice_best_threshold": float(improved_report.get("dice_best_threshold", np.nan)),
    "holdout_dice_best_threshold": float(best_row["dice_mean"]),
    "internal_iou_best_threshold": float(improved_report.get("iou_best_threshold", np.nan)),
    "holdout_iou_best_threshold": float(best_row["iou_mean"]),
}

comparison_df = pd.DataFrame([comparison])
comparison_csv_path = HOLDOUT_ROOT / "E5_holdout_vs_internal_validation.csv"
comparison_df.to_csv(comparison_csv_path, index=False)

print("CSV comparacion:", comparison_csv_path)
display(comparison_df)

## 10. Visualizaciones

In [ ]:
def export_prediction_figure(item, index, threshold=0.5):
    pred = (item["prob"] >= threshold).astype(np.float32)
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))

    axes[0].imshow(item["image"], cmap="gray", vmin=0, vmax=1)
    axes[0].set_title("Imagen")

    axes[1].imshow(item["gt"], cmap="gray", vmin=0, vmax=1)
    axes[1].set_title("Ground truth")

    im = axes[2].imshow(item["prob"], cmap="viridis", vmin=0, vmax=1)
    axes[2].set_title("Probabilidad")
    fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

    axes[3].imshow(pred, cmap="gray", vmin=0, vmax=1)
    axes[3].set_title(f"Pred th={threshold}")

    axes[4].imshow(item["image"], cmap="gray", vmin=0, vmax=1)
    axes[4].imshow(np.ma.masked_where(item["gt"] == 0, item["gt"]), cmap="Greens", alpha=0.45)
    axes[4].imshow(np.ma.masked_where(pred == 0, pred), cmap="Reds", alpha=0.35)
    axes[4].set_title("Overlay GT/pred")

    for ax in axes:
        ax.axis("off")

    fig.suptitle(f"Holdout - {item['case_id']} - slice {item['slice_index']}")
    fig.tight_layout()

    path = FIGURES_ROOT / f"E5_holdout_prediction_example_{index:02d}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    return path


figure_paths = []
for index, item in enumerate(inference_items[:3], start=1):
    figure_paths.append(export_prediction_figure(item, index=index, threshold=0.5))

print("Figuras exportadas:")
for path in figure_paths:
    print(path)

## 11. Reporte JSON

In [ ]:
holdout_report = {
    "t1_candidates_count": int(len(t1_candidates_df)),
    "used_in_notebook_05_count": int(len(used_case_ids)),
    "available_for_holdout_count": int(len(holdout_pool_df)),
    "evaluated_holdout_count": int(len(holdout_df)),
    "thresholds": THRESHOLDS,
    "best_threshold": best_threshold,
    "dice_holdout_threshold_05": float(th05_row["dice_mean"]),
    "iou_holdout_threshold_05": float(th05_row["iou_mean"]),
    "dice_holdout_best_threshold": float(best_row["dice_mean"]),
    "iou_holdout_best_threshold": float(best_row["iou_mean"]),
    "pred_foreground_ratio_mean_threshold_05": float(th05_row["pred_foreground_ratio_mean"]),
    "gt_foreground_ratio_mean_threshold_05": float(th05_row["gt_foreground_ratio_mean"]),
    "sanity_checks": sanity_checks,
    "comparison_against_internal_validation": comparison,
    "exports": {
        "selected_cases": str(holdout_selected_cases_csv_path),
        "metrics_by_case": str(holdout_metrics_csv_path),
        "threshold_summary": str(holdout_summary_csv_path),
        "sanity_checks": str(sanity_checks_json_path),
        "comparison": str(comparison_csv_path),
        "figures": [str(path) for path in figure_paths],
    },
}

holdout_report_json_path = HOLDOUT_ROOT / "E5_holdout_validation_report.json"
holdout_report_json_path.write_text(json.dumps(holdout_report, indent=2), encoding="utf-8")

print("Reporte JSON:", holdout_report_json_path)
print(json.dumps(holdout_report, indent=2))

## 12. Conclusion tecnica Markdown

In [ ]:
if sanity_checks["pct_cases_pred_foreground_ratio_gt_0_8_threshold_05"] > 0:
    sanity_message = "Se observaron casos con prediccion demasiado amplia en threshold 0.5."
elif sanity_checks["pct_cases_pred_foreground_ratio_lt_0_01_threshold_05"] > 0:
    sanity_message = "Se observaron casos con prediccion casi vacia en threshold 0.5."
else:
    sanity_message = "No se detecto colapso evidente a foreground/background en threshold 0.5."

if comparison["holdout_dice_threshold_05"] >= comparison["internal_dice_threshold_05"] - 0.10:
    generalization_message = "El desempeno holdout es razonablemente cercano a la validacion interna."
else:
    generalization_message = "El desempeno holdout cae de forma relevante frente a la validacion interna; conviene revisar robustez."

recommendations = []
recommendations.append("Mantener este holdout como evidencia independiente para comparar proximos experimentos.")
if generalization_message.startswith("El desempeno holdout es razonablemente"):
    recommendations.append("Avanzar a mejorar seleccion de slices, augmentations o evaluar mas datos antes de multiclase.")
else:
    recommendations.append("Antes de multiclase, ampliar datos, revisar split y validar con mas holdout.")
recommendations.append("No pasar todavia a nnU-Net hasta cerrar una comparacion clara del baseline 2D.")

conclusion_md = f"""# Conclusion tecnica - E5 Validacion holdout y sanity checks

## Objetivo

Se evaluo el modelo binario mejorado sobre casos T1 no utilizados en el notebook 05 para verificar si el desempeno generaliza mas alla del split interno.

## Por que se hace holdout

El notebook 05 mostro buen desempeno en validacion interna. Esta validacion holdout busca comprobar que ese resultado no dependa unicamente de los casos usados para entrenar y validar durante el experimento mejorado.

## Configuracion

- Modelo: `{IMPROVED_MODEL_PATH}`
- Candidatos T1: {len(t1_candidates_df)}
- Casos usados notebook 05 excluidos: {len(used_case_ids)}
- Casos disponibles para holdout: {len(holdout_pool_df)}
- Casos evaluados en holdout: {len(holdout_df)}
- Eje sagital: {SAGITTAL_AXIS}
- Target size: {TARGET_SIZE}
- Thresholds evaluados: {THRESHOLDS}

## Resultados holdout

- Dice holdout th 0.5: {holdout_report['dice_holdout_threshold_05']:.4f}
- IoU holdout th 0.5: {holdout_report['iou_holdout_threshold_05']:.4f}
- Mejor threshold: {best_threshold}
- Dice holdout mejor threshold: {holdout_report['dice_holdout_best_threshold']:.4f}
- IoU holdout mejor threshold: {holdout_report['iou_holdout_best_threshold']:.4f}
- Foreground GT promedio th 0.5: {holdout_report['gt_foreground_ratio_mean_threshold_05']:.4f}
- Foreground predicho promedio th 0.5: {holdout_report['pred_foreground_ratio_mean_threshold_05']:.4f}

## Comparacion contra validacion interna

- Dice interno th 0.5: {comparison['internal_dice_threshold_05']:.4f}
- Dice holdout th 0.5: {comparison['holdout_dice_threshold_05']:.4f}
- IoU interno th 0.5: {comparison['internal_iou_threshold_05']:.4f}
- IoU holdout th 0.5: {comparison['holdout_iou_threshold_05']:.4f}
- Interpretacion: {generalization_message}

## Sanity checks

- Mascaras binarias: {sanity_checks['masks_are_binary']}
- Casos con prediccion vacia th 0.5: {len(sanity_checks['empty_prediction_cases_threshold_05'])}
- Casos con prediccion llena th 0.5: {len(sanity_checks['full_prediction_cases_threshold_05'])}
- Porcentaje casos pred_foreground_ratio > 0.8: {sanity_checks['pct_cases_pred_foreground_ratio_gt_0_8_threshold_05']:.4f}
- Porcentaje casos pred_foreground_ratio < 0.01: {sanity_checks['pct_cases_pred_foreground_ratio_lt_0_01_threshold_05']:.4f}
- Interpretacion: {sanity_message}

## Evidencias exportadas

- Casos holdout: `{holdout_selected_cases_csv_path}`
- Metricas por caso: `{holdout_metrics_csv_path}`
- Resumen por threshold: `{holdout_summary_csv_path}`
- Sanity checks: `{sanity_checks_json_path}`
- Comparacion interna vs holdout: `{comparison_csv_path}`
- Reporte JSON: `{holdout_report_json_path}`
- Figuras: {', '.join(str(path) for path in figure_paths)}

## Limitaciones

- Segmentacion binaria.
- Un solo slice sagital por volumen.
- La seleccion del slice todavia usa mascara, por lo que no representa inferencia clinica real.
- No se evalua multiclase, axial, nnU-Net ni backend.

## Recomendacion proximo paso

{chr(10).join("- " + rec for rec in recommendations)}
"""

conclusion_path = DOCS_ROOT / "E5_holdout_validacion_conclusion.md"
conclusion_path.write_text(conclusion_md, encoding="utf-8")

print(conclusion_md)
print("Conclusion Markdown:", conclusion_path)

## 13. Criterio de aceptacion

El notebook es correcto si excluye los casos usados en el notebook 05, evalua el modelo mejorado sin reentrenar, calcula Dice/IoU en holdout, verifica colapso foreground/background, exporta metricas/figuras/JSON/Markdown y deja una conclusion clara sobre generalizacion.